# VLA-0-Smol GPU timing

Settings: **Accelerator = GPU T4** and **Internet = on**. Then choose Run All.

When it finishes, download `gpu_t4.json` from the Output panel into `evidence/benchmark/`.


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q "lerobot[vla0_smol] @ git+https://github.com/Robot-Learning-Collective/lerobot-experiments.git@8e71efbebbb7ea2942b579b1d21e3ec11ac2d271" pyyaml


In [ ]:
import json, pathlib
files = json.loads("{\"src/load_policy.py\": \"\\\"\\\"\\\"Load VLA-0-Smol with every Hugging Face revision pinned. Plumbing, no research logic.\\\"\\\"\\\"\\nfrom pathlib import Path\\n\\nimport torch\\nimport yaml\\nfrom huggingface_hub import snapshot_download\\nfrom safetensors.torch import load_file\\n\\nfrom lerobot.configs.policies import PreTrainedConfig\\nfrom lerobot.policies.factory import make_pre_post_processors\\nfrom lerobot.policies.vla0_smol.modeling_vla0_smol import VLA0SmolPolicy\\n\\n\\ndef load_config(path):\\n    return yaml.safe_load(Path(path).read_text())\\n\\n\\ndef fetch_pinned(pins):\\n    \\\"\\\"\\\"Download the exact revisions and return their local folders.\\\"\\\"\\\"\\n    model_dir = snapshot_download(pins[\\\"model\\\"], revision=pins[\\\"model_revision\\\"])\\n    # VLA0.__init__ loads vlm_checkpoint with no revision, so hand it a pinned local folder.\\n    base_dir = snapshot_download(\\n        pins[\\\"base_vlm\\\"],\\n        revision=pins[\\\"base_vlm_revision\\\"],\\n        allow_patterns=[\\\"*.json\\\", \\\"*.safetensors\\\", \\\"*.txt\\\", \\\"*.jinja\\\"],\\n    )\\n    return model_dir, base_dir\\n\\n\\ndef check_weights_loaded(policy, model_dir):\\n    \\\"\\\"\\\"from_pretrained uses strict=False, which would skip a mismatched key without an error.\\n\\n    Compare every checkpoint tensor with the live model. Returns a dict of counts.\\n    \\\"\\\"\\\"\\n    ckpt = load_file(str(Path(model_dir) / \\\"model.safetensors\\\"))\\n    live = policy.state_dict()\\n    missing_in_model = [k for k in ckpt if k not in live]\\n    not_in_ckpt = [k for k in live if k not in ckpt]\\n    mismatched = [\\n        k\\n        for k in ckpt\\n        if k in live and not torch.equal(ckpt[k].to(\\\"cpu\\\"), live[k].detach().to(\\\"cpu\\\"))\\n    ]\\n    return {\\n        \\\"checkpoint_tensors\\\": len(ckpt),\\n        \\\"model_tensors\\\": len(live),\\n        \\\"in_checkpoint_not_model\\\": missing_in_model,\\n        \\\"in_model_not_checkpoint\\\": not_in_ckpt,\\n        \\\"values_differ\\\": mismatched,\\n    }\\n\\n\\ndef load_policy(pins, device=\\\"cpu\\\"):\\n    \\\"\\\"\\\"Returns (policy, preprocessor, postprocessor, model_dir).\\\"\\\"\\\"\\n    model_dir, base_dir = fetch_pinned(pins)\\n\\n    cfg = PreTrainedConfig.from_pretrained(model_dir)\\n    cfg.device = device  # the saved config says \\\"cuda\\\"\\n    cfg.use_amp = False\\n    cfg.vlm_checkpoint = base_dir\\n\\n    policy = VLA0SmolPolicy.from_pretrained(model_dir, config=cfg)\\n    policy.eval()\\n\\n    pre, post = make_pre_post_processors(\\n        policy_cfg=cfg,\\n        pretrained_path=model_dir,\\n        preprocessor_overrides={\\\"device_processor\\\": {\\\"device\\\": device}},\\n        postprocessor_overrides={\\\"device_processor\\\": {\\\"device\\\": \\\"cpu\\\"}},\\n    )\\n    return policy, pre, post, model_dir\\n\", \"src/snapshots.py\": \"\\\"\\\"\\\"Fetch recorded demo frames and turn them into exact model inputs. Plumbing, no research logic.\\\"\\\"\\\"\\nimport io\\n\\nimport numpy as np\\nimport pyarrow.parquet as pq\\nimport torch\\nfrom huggingface_hub import HfFileSystem, hf_hub_download\\nfrom PIL import Image\\n\\nIMAGE_KEYS = (\\\"observation.images.image\\\", \\\"observation.images.image2\\\")\\n\\n\\ndef _dataset_file(pins, filename):\\n    return hf_hub_download(\\n        pins[\\\"dataset\\\"], filename, repo_type=\\\"dataset\\\", revision=pins[\\\"dataset_revision\\\"]\\n    )\\n\\n\\ndef _to_chw_float(cell):\\n    \\\"\\\"\\\"Stored PNG bytes -> CHW float tensor in [0, 1], the form the policy expects.\\\"\\\"\\\"\\n    array = np.array(Image.open(io.BytesIO(cell[\\\"bytes\\\"])).convert(\\\"RGB\\\"))\\n    return torch.from_numpy(array).permute(2, 0, 1).float() / 255.0\\n\\n\\ndef _index_range(fs, pins, name):\\n    \\\"\\\"\\\"Read the min and max of the `index` column from a file's footer, without downloading it.\\\"\\\"\\\"\\n    path = f\\\"datasets/{pins['dataset']}@{pins['dataset_revision']}/{name}\\\"\\n    with fs.open(path) as handle:\\n        meta = pq.ParquetFile(handle).metadata\\n        column = meta.schema.names.index(\\\"index\\\")\\n        stats = [meta.row_group(i).column(column).statistics for i in range(meta.num_row_groups)]\\n    return min(s.min for s in stats), max(s.max for s in stats)\\n\\n\\ndef _find_data_file(fs, pins, names, global_index):\\n    \\\"\\\"\\\"Binary search by each file's real index range.\\n\\n    meta/episodes data/file_index is wrong at this dataset revision (it puts episode 379 in\\n    file-025, which actually holds episodes 61-63), so it isn't used.\\n    \\\"\\\"\\\"\\n    low, high = 0, len(names) - 1\\n    while low <= high:\\n        middle = (low + high) // 2\\n        first, last = _index_range(fs, pins, names[middle])\\n        if global_index < first:\\n            high = middle - 1\\n        elif global_index > last:\\n            low = middle + 1\\n        else:\\n            return names[middle]\\n    raise LookupError(f\\\"no data file holds index {global_index}\\\")\\n\\n\\ndef load_frames(pins, snapshots):\\n    \\\"\\\"\\\"One frame per episode, from the first n_episodes of the task in episode order.\\\"\\\"\\\"\\n    episodes = pq.read_table(_dataset_file(pins, \\\"meta/episodes/chunk-000/file-000.parquet\\\")).to_pandas()\\n    tasks = pq.read_table(_dataset_file(pins, \\\"meta/tasks.parquet\\\")).to_pandas()\\n    task_index = int(tasks.loc[snapshots[\\\"task\\\"], \\\"task_index\\\"])\\n    chosen = episodes[episodes[\\\"tasks\\\"].apply(lambda tasks: snapshots[\\\"task\\\"] in list(tasks))]\\n    chosen = chosen.sort_values(\\\"episode_index\\\").head(snapshots[\\\"n_episodes\\\"])\\n\\n    fs = HfFileSystem()\\n    root = f\\\"datasets/{pins['dataset']}@{pins['dataset_revision']}/\\\"\\n    names = sorted(\\n        path.split(\\\"@\\\", 1)[1].split(\\\"/\\\", 1)[1] for path in fs.glob(root + \\\"data/*/*.parquet\\\")\\n    )\\n\\n    frames = []\\n    for _, episode in chosen.iterrows():\\n        offset = int(snapshots[\\\"frame_fraction\\\"] * int(episode[\\\"length\\\"]))\\n        global_index = int(episode[\\\"dataset_from_index\\\"]) + offset\\n        name = _find_data_file(fs, pins, names, global_index)\\n        table = pq.read_table(_dataset_file(pins, name), filters=[(\\\"index\\\", \\\"=\\\", global_index)]).to_pylist()\\n        assert len(table) == 1, f\\\"expected one row for index {global_index}, got {len(table)}\\\"\\n        row = table[0]\\n        assert row[\\\"episode_index\\\"] == int(episode[\\\"episode_index\\\"]), (global_index, row[\\\"episode_index\\\"])\\n        assert row[\\\"frame_index\\\"] == offset, (global_index, row[\\\"frame_index\\\"], offset)\\n        assert row[\\\"task_index\\\"] == task_index, (global_index, row[\\\"task_index\\\"], task_index)\\n        frames.append(\\n            {\\n                \\\"episode_index\\\": int(episode[\\\"episode_index\\\"]),\\n                \\\"frame_index\\\": int(row[\\\"frame_index\\\"]),\\n                \\\"observation.images.image\\\": _to_chw_float(row[IMAGE_KEYS[0]]),\\n                \\\"observation.images.image2\\\": _to_chw_float(row[IMAGE_KEYS[1]]),\\n                \\\"observation.state\\\": torch.tensor(row[\\\"observation.state\\\"], dtype=torch.float32),\\n                \\\"recorded_action\\\": row[\\\"action\\\"],\\n            }\\n        )\\n    return frames\\n\\n\\ndef make_batch(frame, instruction, preprocessor):\\n    \\\"\\\"\\\"Same image and state, chosen instruction, through the checkpoint's own normalizer.\\\"\\\"\\\"\\n    raw = {key: frame[key].clone() for key in (*IMAGE_KEYS, \\\"observation.state\\\")}\\n    raw[\\\"task\\\"] = instruction\\n    return preprocessor(raw)\\n\\n\\ndef prompt_inputs(policy, batch):\\n    \\\"\\\"\\\"The exact tokens and pixels generate_actions would feed the model, with no action text.\\\"\\\"\\\"\\n    model = policy.model\\n    inputs, _ = model.create_input_tokens(\\n        states=batch[\\\"observation.state\\\"],\\n        images=model.prepare_images(batch),\\n        lang_text=batch[\\\"task\\\"],\\n        actions=None,\\n    )\\n    return inputs\\n\", \"scripts/benchmark.py\": \"\\\"\\\"\\\"Measure forward-pass time, generation time and peak memory on one device. Run once per device.\\\"\\\"\\\"\\nimport argparse\\nimport json\\nimport os\\nimport platform\\nimport resource\\nimport statistics\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nimport torch\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \\\"src\\\"))\\nfrom load_policy import load_config, load_policy\\nfrom snapshots import load_frames, make_batch, prompt_inputs\\n\\nparser = argparse.ArgumentParser()\\nparser.add_argument(\\\"--config\\\", default=\\\"config/smoke.yaml\\\")\\nparser.add_argument(\\\"--device\\\", default=\\\"cpu\\\")\\nparser.add_argument(\\\"--forwards\\\", type=int, default=20)\\nparser.add_argument(\\\"--generations\\\", type=int, default=10)\\nparser.add_argument(\\\"--out\\\", required=True)\\nargs = parser.parse_args()\\ncfg = load_config(args.config)\\n\\n\\ndef sync():\\n    if args.device == \\\"cuda\\\":\\n        torch.cuda.synchronize()\\n    elif args.device == \\\"mps\\\":\\n        torch.mps.synchronize()\\n\\n\\ndef peak_rss_gb():\\n    # ru_maxrss is bytes on macOS and kilobytes on Linux.\\n    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\\n    return raw / 1e9 if sys.platform == \\\"darwin\\\" else raw * 1024 / 1e9\\n\\n\\ndef summary(values):\\n    return {\\n        \\\"n\\\": len(values),\\n        \\\"median_s\\\": round(statistics.median(values), 4),\\n        \\\"mean_s\\\": round(statistics.mean(values), 4),\\n        \\\"min_s\\\": round(min(values), 4),\\n        \\\"max_s\\\": round(max(values), 4),\\n        \\\"raw_s\\\": [round(v, 4) for v in values],\\n    }\\n\\n\\nstart = time.perf_counter()\\npolicy, pre, post, _ = load_policy(cfg[\\\"pins\\\"], device=args.device)\\nload_seconds = time.perf_counter() - start\\n\\nframe = load_frames(cfg[\\\"pins\\\"], cfg[\\\"snapshots\\\"])[0]\\nbatches = [make_batch(frame, text, pre) for text in cfg[\\\"instructions\\\"].values()]\\ninputs = [prompt_inputs(policy, batch) for batch in batches]\\n\\nmps_peak = 0\\n\\n\\ndef sample_device_memory():\\n    global mps_peak\\n    if args.device == \\\"mps\\\":\\n        mps_peak = max(mps_peak, torch.mps.driver_allocated_memory())\\n\\n\\n# The first pass of each kind is timed separately because it includes warm-up.\\nforward_times = []\\nfor i in range(args.forwards + 1):\\n    start = time.perf_counter()\\n    with torch.no_grad():\\n        policy.model.vlm(**inputs[i % 2], use_cache=False)\\n    sync()\\n    forward_times.append(time.perf_counter() - start)\\n    sample_device_memory()\\n\\ngenerate_times, fallbacks = [], 0\\nfor i in range(args.generations + 1):\\n    start = time.perf_counter()\\n    with torch.no_grad():\\n        actions = policy.model.generate_actions(batches[i % 2])\\n    sync()\\n    generate_times.append(time.perf_counter() - start)\\n    sample_device_memory()\\n    fallbacks += int((actions < -0.99).all())\\n\\nresult = {\\n    \\\"device\\\": args.device,\\n    \\\"machine\\\": platform.platform(),\\n    \\\"processor\\\": platform.processor(),\\n    \\\"cuda_device\\\": torch.cuda.get_device_name(0) if args.device == \\\"cuda\\\" else None,\\n    \\\"mps_cpu_fallback\\\": os.environ.get(\\\"PYTORCH_ENABLE_MPS_FALLBACK\\\"),  # image resize only\\n    \\\"torch\\\": torch.__version__,\\n    \\\"episode_index\\\": frame[\\\"episode_index\\\"],\\n    \\\"frame_index\\\": frame[\\\"frame_index\\\"],\\n    \\\"load_seconds\\\": round(load_seconds, 2),\\n    \\\"forward_warmup_s\\\": round(forward_times[0], 4),\\n    \\\"forward\\\": summary(forward_times[1:]),\\n    \\\"generate_warmup_s\\\": round(generate_times[0], 4),\\n    \\\"generate\\\": summary(generate_times[1:]),\\n    \\\"bin_zero_fallbacks\\\": fallbacks,\\n    \\\"peak_process_rss_gb\\\": round(peak_rss_gb(), 2),\\n    \\\"peak_mps_driver_allocated_gb\\\": round(mps_peak / 1e9, 2) if args.device == \\\"mps\\\" else None,\\n    \\\"peak_cuda_allocated_gb\\\": round(torch.cuda.max_memory_allocated() / 1e9, 2) if args.device == \\\"cuda\\\" else None,\\n}\\nout = Path(args.out)\\nout.parent.mkdir(parents=True, exist_ok=True)\\nout.write_text(json.dumps(result, indent=2))\\nprint(json.dumps({k: v for k, v in result.items() if k not in (\\\"forward\\\", \\\"generate\\\")}, indent=2))\\nprint(\\\"forward\\\", {k: v for k, v in result[\\\"forward\\\"].items() if k != \\\"raw_s\\\"})\\nprint(\\\"generate\\\", {k: v for k, v in result[\\\"generate\\\"].items() if k != \\\"raw_s\\\"})\\n\", \"config/smoke.yaml\": \"# Stage 0 smoke run: prompt-only forward pass, no generation, no simulator.\\nname: smoke\\npins: &pins\\n  lerobot_commit: 8e71efbebbb7ea2942b579b1d21e3ec11ac2d271   # lerobot-experiments, dev\\n  model: Robot-Learning-Collective/VLA-0-Smol\\n  model_revision: 551fe3d5d3debb6532f0675a7a68b3b245ead4d8\\n  base_vlm: HuggingFaceTB/SmolVLM2-500M-Video-Instruct       # processor and tokenizer come from here\\n  base_vlm_revision: 7b375e1b73b11138ff12fe22c8f2822d8fe03467\\n  dataset: HuggingFaceVLA/libero\\n  dataset_revision: 86958911c0f959db2bbbdb107eb3e17c5f9c798e\\n\\n# Recorded demo frames, not saved simulator states. Frame position is a placeholder\\n# and is NOT checked to be after the bowl is lifted. Stage 2 replaces this.\\nsnapshots:\\n  source: dataset_frame\\n  task: put the bowl on the plate\\n  n_episodes: 1\\n  frame_fraction: 0.6\\n\\ninstructions:\\n  plate: put the bowl on the plate\\n  stove: put the bowl on the stove\\n\\nreadout:\\n  words: [plate, stove]   # exact tokens get checked and logged by the run\\n  layers: all\\n  positions: [prompt_end]\\n\\nchecks:\\n  self_patch_identity: true\\n  self_patch_layers: all\\n\\ndevice: cpu\\nseed: 0\\n\"}")
for name, text in files.items():
    path = pathlib.Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
print(sorted(files))


In [ ]:
!python scripts/benchmark.py --device cuda --forwards 20 --generations 50 --out /kaggle/working/gpu_t4.json


In [ ]:
!pip freeze > /kaggle/working/gpu_t4_pip_freeze.txt
!cat /kaggle/working/gpu_t4.json
